# Full CA Total R² simulation source

Research code from my MSc thesis, University of Copenhagen, 2024.

Original research cells are preserved; saved outputs are cleared. See the repository README and docs/source_notes.md for inputs and execution context.


In [ ]:
import numpy as np
import pandas as pd

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
R2_target = 0.45  # Target R2 for calibration

# Simulate characteristics
rho = np.random.uniform(0.9, 1, Pc)
c = np.zeros((N, Pc, T))
for t in range(1, T):
    c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# Normalize characteristics
c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# Generate factors and errors
x = np.random.normal(0.03, 0.12, (Px, T))
eta = np.random.normal(0, 0.01, (3, T))
epsilon = np.random.standard_t(5, (N, T))

# Define linear factor loadings
beta_linear = np.zeros((N, 3, T))
for t in range(1, T):
    beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                     c_normalized[:, 1, t],
                                     0.8 * c_normalized[:, 2, t]]).T



# Initial Returns Calculation
f = np.dot(np.random.randn(3, Px), x) + eta
r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:] # Slice epsilon to match dimensions
# Calculate initial R2
explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
total_variance = np.var(r, axis=1)
initial_R2 = np.mean(explained_variance / total_variance)
print(f'Initial R2: {initial_R2:.2f}')

# Adjust the variances to achieve the target R2
adjustment_factor = np.sqrt(R2_target / initial_R2)
f_adjusted = adjustment_factor * f
epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

# Generate adjusted returns
r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:,1:]

# Calculate adjusted R2
explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
total_variance_adjusted = np.var(r_adjusted, axis=1)
adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
print(f'Adjusted R2: {adjusted_R2:.2f}')

# Split data into training, validation, and testing sets
train_ratio = 0.6
val_ratio = 0.2
train_size = int(T * train_ratio)
val_size = int(T * val_ratio)

train_data = r_adjusted[:, :train_size]
val_data = r_adjusted[:, train_size:train_size+val_size]
test_data = r_adjusted[:, train_size+val_size:]

# Print the shapes of the datasets
print("Training Data Shape:", train_data.shape)
print("Validation Data Shape:", val_data.shape)
print("Testing Data Shape:", test_data.shape)

# Example plot to visualize the generated data
import matplotlib.pyplot as plt

plt.plot(np.mean(train_data, axis=0), label='Train Data')
plt.plot(np.mean(val_data, axis=0), label='Validation Data')
plt.plot(np.mean(test_data, axis=0), label='Test Data')
plt.legend()
plt.title('Average Returns Over Time')
plt.xlabel('Time')
plt.ylabel('Average Return')
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
R2_target = 0.45  # Target R2 for calibration

# Simulate characteristics
rho = np.random.uniform(0.9, 1, Pc)
c = np.zeros((N, Pc, T))
for t in range(1, T):
    c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# Normalize characteristics
c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# Generate factors and errors
x = np.random.normal(0.03, 0.12, (Px, T))
eta = np.random.normal(0, 0.01, (3, T))
epsilon = np.random.standard_t(5, (N, T))

# Define linear factor loadings
beta_linear = np.zeros((N, 3, T))
for t in range(1, T):
    beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t]*c_normalized[:, 2, t],
                                     c_normalized[:, 1, t]**3,
                                     0.8 * c_normalized[:, 2, t]*c_normalized[:, 0, t]*c_normalized[:, 1, t]]).T



# Initial Returns Calculation
f = np.dot(np.random.randn(3, Px), x) + eta
r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:] # Slice epsilon to match dimensions
# Calculate initial R2
explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
total_variance = np.var(r, axis=1)
initial_R2 = np.mean(explained_variance / total_variance)
print(f'Initial R2: {initial_R2:.2f}')

# Adjust the variances to achieve the target R2
adjustment_factor = np.sqrt(R2_target / initial_R2)
f_adjusted = adjustment_factor * f
epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

# Generate adjusted returns
r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:,1:]

# Calculate adjusted R2
explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
total_variance_adjusted = np.var(r_adjusted, axis=1)
adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
print(f'Adjusted R2: {adjusted_R2:.2f}')

# Split data into training, validation, and testing sets
train_ratio = 0.6
val_ratio = 0.2
train_size = int(T * train_ratio)
val_size = int(T * val_ratio)

train_data = r_adjusted[:, :train_size]
val_data = r_adjusted[:, train_size:train_size+val_size]
test_data = r_adjusted[:, train_size+val_size:]

# Print the shapes of the datasets
print("Training Data Shape:", train_data.shape)
print("Validation Data Shape:", val_data.shape)
print("Testing Data Shape:", test_data.shape)

# Example plot to visualize the generated data
import matplotlib.pyplot as plt

plt.plot(np.mean(train_data, axis=0), label='Train Data')
plt.plot(np.mean(val_data, axis=0), label='Validation Data')
plt.plot(np.mean(test_data, axis=0), label='Test Data')
plt.legend()
plt.title('Average Returns Over Time')
plt.xlabel('Time')
plt.ylabel('Average Return')
plt.show()


In [ ]:
import numpy as np
import pandas as pd

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

# Simulate characteristics
rho = np.random.uniform(0.9, 1, Pc)
c = np.zeros((N, Pc, T))
for t in range(1, T):
    c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

# Normalize characteristics
c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

# Generate factors and errors
x = np.random.normal(0.03, 0.12, (Px, T))
eta = np.random.normal(0, 0.01, (num_factors, T))
epsilon = np.random.standard_t(5, (N, T))

# Define linear factor loadings
beta_linear = np.zeros((N, num_factors, T))
for t in range(1, T):
    beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                     c_normalized[:, 1, t],
                                     0.8 * c_normalized[:, 2, t],
                                     0.6 * c_normalized[:, 3, t],
                                     0.4 * c_normalized[:, 4, t]]).T

# Initial Returns Calculation
f = np.dot(np.random.randn(num_factors, Px), x) + eta
r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:] # Slice epsilon to match dimensions

# Calculate initial R2
explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
total_variance = np.var(r, axis=1)
initial_R2 = np.mean(explained_variance / total_variance)
print(f'Initial R2: {initial_R2:.2f}')

# Adjust the variances to achieve the target R2
adjustment_factor = np.sqrt(R2_target / initial_R2)
f_adjusted = adjustment_factor * f
epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

# Generate adjusted returns
r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

# Calculate adjusted R2
explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
total_variance_adjusted = np.var(r_adjusted, axis=1)
adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)
print(f'Adjusted R2: {adjusted_R2:.2f}')

# Split data into training, validation, and testing sets
train_ratio = 0.6
val_ratio = 0.2
train_size = int(T * train_ratio)
val_size = int(T * val_ratio)

train_data = r_adjusted[:, :train_size]
val_data = r_adjusted[:, train_size:train_size+val_size]
test_data = r_adjusted[:, train_size+val_size:]

# Print the shapes of the datasets
print("Training Data Shape:", train_data.shape)
print("Validation Data Shape:", val_data.shape)
print("Testing Data Shape:", test_data.shape)

# Example plot to visualize the generated data
import matplotlib.pyplot as plt

plt.plot(np.mean(train_data, axis=0), label='Train Data')
plt.plot(np.mean(val_data, axis=0), label='Validation Data')
plt.plot(np.mean(test_data, axis=0), label='Test Data')
plt.legend()
plt.title('Average Returns Over Time')
plt.xlabel('Time')
plt.ylabel('Average Return')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2

# Monte Carlo Simulation
num_simulations = 100
results = []

for i in tqdm(range(num_simulations)):
    train_data, val_data, test_data, adjusted_R2 = generate_data(N, T, Pc, Px, num_factors, R2_target)
    results.append(adjusted_R2)
    if (i + 1) % 10 == 0:
        print(f'Simulation {i + 1}/{num_simulations} completed')

# Aggregate Results
mean_R2 = np.mean(results)
std_R2 = np.std(results)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_R2:.4f}')

# Plotting the distribution of R2
plt.hist(results, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from tqdm import tqdm

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2

def fit_pca_model(train_data, num_factors):
    pca = PCA(n_components=num_factors)
    pca.fit(train_data.T)
    return pca

def evaluate_model(pca, data):
    transformed_data = pca.transform(data.T)
    reconstructed_data = pca.inverse_transform(transformed_data)
    residuals = data - reconstructed_data.T
    explained_variance = np.var(reconstructed_data.T, axis=1)
    total_variance = np.var(data, axis=1)
    R2 = np.mean(explained_variance / total_variance)
    return R2

# Monte Carlo Simulation
num_simulations = 100
results = []

for i in tqdm(range(num_simulations)):
    train_data, val_data, test_data, adjusted_R2 = generate_data(N, T, Pc, Px, num_factors, R2_target)

    # Fit PCA model
    pca_model = fit_pca_model(train_data, num_factors)

    # Evaluate PCA model on validation data
    val_R2 = evaluate_model(pca_model, val_data)

    # Evaluate PCA model on test data
    test_R2 = evaluate_model(pca_model, test_data)

    results.append((adjusted_R2, val_R2, test_R2))

    if (i + 1) % 10 == 0:
        print(f'Simulation {i + 1}/{num_simulations} completed')

# Aggregate Results
adjusted_R2_values = [res[0] for res in results]
val_R2_values = [res[1] for res in results]
test_R2_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2 = np.mean(val_R2_values)
std_val_R2 = np.std(val_R2_values)

mean_test_R2 = np.mean(test_R2_values)
std_test_R2 = np.std(test_R2_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 over {num_simulations} simulations: {mean_val_R2:.4f}')
print(f'Standard Deviation of Validation R2: {std_val_R2:.4f}')

print(f'Mean Test R2 over {num_simulations} simulations: {mean_test_R2:.4f}')
print(f'Standard Deviation of Test R2: {std_test_R2:.4f}')

# Plotting the distribution of Adjusted R2
plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Validation R2
plt.hist(val_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2')
plt.xlabel('Validation R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Test R2
plt.hist(test_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2')
plt.xlabel('Test R2')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, f_adjusted

def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def make_model(N, Pc, n_factors=6, return_components=False, l1_reg=0.01):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model
    else:
        return model

# Monte Carlo Simulation
num_simulations = 100
results = []

for i in tqdm(range(num_simulations)):
    train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, f_adjusted = generate_data(N, T, Pc, Px, num_factors, R2_target)

    # Combine data for the entire time period
    combined_data = np.concatenate([train_data, val_data, test_data], axis=1)

    Z_t_minus_1_all = np.array([c_normalized[:, :, t-1] for t in range(1, T-1)])
    r_t_all = np.array([combined_data[:, t] for t in range(1, T-1)])

    factor_network_input_all = calculate_factor_network_input(Z_t_minus_1_all, r_t_all)

    # Lag the factors
    input_factor_all = factor_network_input_all[:-1]
    r_t_all_lagged = combined_data[:, 1:T-1].T

    main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, return_components=True, l1_reg=0.001)

    # Split into training, validation, and testing sets
    train_size_lagged = train_size - 1
    val_size_lagged = len(val_data[0]) - 1
    test_size_lagged = len(test_data[0]) - 1

    input_factor_train = input_factor_all[:train_size_lagged]
    input_factor_val = input_factor_all[train_size_lagged:train_size_lagged + val_size_lagged]
    input_factor_test = input_factor_all[train_size_lagged + val_size_lagged:train_size_lagged + val_size_lagged + test_size_lagged]

    Z_t_minus_1_train = Z_t_minus_1_all[:train_size_lagged]
    Z_t_minus_1_val = Z_t_minus_1_all[train_size_lagged:train_size_lagged + val_size_lagged]
    Z_t_minus_1_test = Z_t_minus_1_all[train_size_lagged + val_size_lagged:train_size_lagged + val_size_lagged + test_size_lagged]

    main_model.fit([Z_t_minus_1_train, input_factor_train], r_t_all_lagged[:train_size_lagged], epochs=100, batch_size=32, verbose=0)

    val_pred = main_model.predict([Z_t_minus_1_val, input_factor_val])
    val_R2_nn = np.mean(1 - np.var(val_data[:, 1:].T - val_pred, axis=1) / np.var(val_data[:, 1:].T, axis=1))
    print(f'VAL R^2 : {val_R2_nn}')

    test_pred = main_model.predict([Z_t_minus_1_test, input_factor_test])
    test_R2_nn = np.mean(1 - np.var(test_data[:, 1:].T - test_pred, axis=1) / np.var(test_data[:, 1:].T, axis=1))
    print(f'TEST R^2 : {test_R2_nn}')
    results.append((adjusted_R2, val_R2_nn, test_R2_nn))

    if (i + 1) % 10 == 0:
        print(f'Simulation {i + 1}/{num_simulations} completed')

adjusted_R2_values = [res[0] for res in results]
val_R2_nn_values = [res[1] for res in results]
test_R2_nn_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2_nn = np.mean(val_R2_nn_values)
std_val_R2_nn = np.std(val_R2_nn_values)

mean_test_R2_nn = np.mean(test_R2_nn_values)
std_test_R2_nn = np.std(test_R2_nn_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2 (NN)')
plt.xlabel('Validation R2 (NN)')
plt.ylabel('Frequency')
plt.show()

plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2 (NN)')
plt.xlabel('Test R2 (NN)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping # Import EarlyStopping from tensorflow.keras.callbacks

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# callback = EarlyStopping(monitor='loss',patience=3)
# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]



def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def make_model(N, Pc, n_factors=6, l1_reg=0.01, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# Monte Carlo Simulation with L1 tuning
num_simulations = 100
results = []

for i in tqdm(range(num_simulations)):
    train_data, val_data, test_data, adjusted_R2, c_normalized, train_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

    # Prepare inputs for the factor network input calculation
    Z_t_minus_1_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size)])
    r_t_train = np.array([train_data[:, t] for t in range(1, train_size)])

    # Calculate the input for the factor network
    factor_network_input_train = calculate_factor_network_input(Z_t_minus_1_train, r_t_train)

    best_val_R2_nn = -np.inf
    best_l1_value = None

    for l1_value in l1_values:
        # Build and train the neural network model with current l1_value
        main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, return_components=True)

        # Prepare inputs for the model
        input_beta_train = Z_t_minus_1_train
        input_factor_train = factor_network_input_train

        # Train the model
        main_model.fit([input_beta_train, input_factor_train], train_data[:, 1:].T, epochs=100, batch_size=32, verbose=1, callbacks =[early_stopping])

        # Prepare validation inputs
        Z_t_minus_1_val = np.array([c_normalized[:, :, t-1] for t in range(train_size, train_size + val_data.shape[1])])
        Z_t_minus_1_val = Z_t_minus_1_val[:val_data.shape[1]]  # Ensure the shape is consistent
        val_input_beta = Z_t_minus_1_val.reshape(val_data.shape[1], N, Pc)

        val_r_t = val_data.T  # Use the returns for validation
        val_r_t = val_r_t[:val_input_beta.shape[0]]  # Ensure the shape is consistent

        # Prepare validation factor input
        val_factor_network_input = calculate_factor_network_input(val_input_beta, val_r_t)

        # Evaluate on validation data
        val_pred = main_model.predict([val_input_beta, val_factor_network_input])
        val_R2_nn = np.mean(1 - np.var(val_r_t.T - val_pred.T, axis=1) / np.var(val_r_t.T, axis=1))

        # Select the best l1_value based on validation performance
        if val_R2_nn > best_val_R2_nn:
            best_val_R2_nn = val_R2_nn
            best_l1_value = l1_value

    # Train final model with the best l1_value on the entire training data
    main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, return_components=True)
    main_model.fit([input_beta_train, input_factor_train], train_data[:, 1:].T, epochs=100, batch_size=32, verbose=1, callbacks =[early_stopping])

    # Prepare test inputs
    Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_data.shape[1], T)])
    Z_t_minus_1_test = Z_t_minus_1_test[:test_data.shape[1]]  # Ensure the shape is consistent
    test_input_beta = Z_t_minus_1_test.reshape(test_data.shape[1], N, Pc)

    test_r_t = test_data.T  # Use the returns for testing
    test_r_t = test_r_t[:test_input_beta.shape[0]]  # Ensure the shape is consistent

    # Prepare test factor input
    test_factor_network_input = calculate_factor_network_input(test_input_beta, test_r_t)

    # Evaluate on test data
    test_pred = main_model.predict([test_input_beta, test_factor_network_input])
    test_R2_nn = np.mean(1 - np.var(test_r_t.T - test_pred.T, axis=1) / np.var(test_r_t.T, axis=1))

    results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))

    if (i + 1) % 10 == 0:
        print(f'Simulation {i + 1}/{num_simulations} completed')

# Aggregate Results
adjusted_R2_values = [res[0] for res in results]
val_R2_nn_values = [res[1] for res in results]
test_R2_nn_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2_nn = np.mean(val_R2_nn_values)
std_val_R2_nn = np.std(val_R2_nn_values)

mean_test_R2_nn = np.mean(test_R2_nn_values)
std_test_R2_nn = np.std(test_R2_nn_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

# Plotting the distribution of Adjusted R2
plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Validation R2 (NN)
plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2 (NN)')
plt.xlabel('Validation R2 (NN)')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Test R2 (NN)
plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2 (NN)')
plt.xlabel('Test R2 (NN)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping # Import EarlyStopping from tensorflow.keras.callbacks

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

early_stopping = EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True)

# callback = EarlyStopping(monitor='loss',patience=3)
# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]



def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

def make_model(N, Pc, n_factors=6, l1_reg=0.01, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

# Monte Carlo Simulation with L1 tuning
num_simulations = 100
results = []

for i in tqdm(range(num_simulations)):
    train_data, val_data, test_data, adjusted_R2, c_normalized, train_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

    # Prepare inputs for the factor network input calculation
    Z_t_minus_1_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size)])
    r_t_train = np.array([train_data[:, t] for t in range(1, train_size)])

    # Calculate the input for the factor network
    factor_network_input_train = calculate_factor_network_input(Z_t_minus_1_train, r_t_train)

    best_val_R2_nn = -np.inf
    best_l1_value = None

    for l1_value in l1_values:
        # Build and train the neural network model with current l1_value
        main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, return_components=True)

        # Prepare inputs for the model
        input_beta_train = Z_t_minus_1_train
        input_factor_train = factor_network_input_train

        # Prepare validation inputs
        Z_t_minus_1_val = np.array([c_normalized[:, :, t-1] for t in range(train_size, train_size + val_data.shape[1])])
        Z_t_minus_1_val = Z_t_minus_1_val[:val_data.shape[1]]  # Ensure the shape is consistent
        val_input_beta = Z_t_minus_1_val.reshape(val_data.shape[1], N, Pc)

        val_r_t = val_data.T  # Use the returns for validation
        val_r_t = val_r_t[:val_input_beta.shape[0]]  # Ensure the shape is consistent

        # Prepare validation factor input
        val_factor_network_input = calculate_factor_network_input(val_input_beta, val_r_t)

        # Train the model
        main_model.fit([input_beta_train, input_factor_train], train_data[:, 1:].T,validation_data = ([val_input_beta, val_factor_network_input,val_data[:, 1:].T]), epochs=100, batch_size=32, verbose=1, callbacks =[early_stopping])


        # Evaluate on validation data
        val_pred = main_model.predict([val_input_beta, val_factor_network_input])
        val_R2_nn = np.mean(1 - np.var(val_r_t.T - val_pred.T, axis=1) / np.var(val_r_t.T, axis=1))

        # Select the best l1_value based on validation performance
        if val_R2_nn > best_val_R2_nn:
            best_val_R2_nn = val_R2_nn
            best_l1_value = l1_value

    # Train final model with the best l1_value on the entire training data
    main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, return_components=True)
    main_model.fit([input_beta_train, input_factor_train], train_data[:, 1:].T, epochs=100, batch_size=32, verbose=1, callbacks =[early_stopping])

    # Prepare test inputs
    Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_data.shape[1], T)])
    Z_t_minus_1_test = Z_t_minus_1_test[:test_data.shape[1]]  # Ensure the shape is consistent
    test_input_beta = Z_t_minus_1_test.reshape(test_data.shape[1], N, Pc)

    test_r_t = test_data.T  # Use the returns for testing
    test_r_t = test_r_t[:test_input_beta.shape[0]]  # Ensure the shape is consistent

    # Prepare test factor input
    test_factor_network_input = calculate_factor_network_input(test_input_beta, test_r_t)

    # Evaluate on test data
    test_pred = main_model.predict([test_input_beta, test_factor_network_input])
    test_R2_nn = np.mean(1 - np.var(test_r_t.T - test_pred.T, axis=1) / np.var(test_r_t.T, axis=1))

    results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))

    if (i + 1) % 10 == 0:
        print(f'Simulation {i + 1}/{num_simulations} completed')

# Aggregate Results
adjusted_R2_values = [res[0] for res in results]
val_R2_nn_values = [res[1] for res in results]
test_R2_nn_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2_nn = np.mean(val_R2_nn_values)
std_val_R2_nn = np.std(val_R2_nn_values)

mean_test_R2_nn = np.mean(test_R2_nn_values)
std_test_R2_nn = np.std(test_R2_nn_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

# Plotting the distribution of Adjusted R2
plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Validation R2 (NN)
plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2 (NN)')
plt.xlabel('Validation R2 (NN)')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Test R2 (NN)
plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2 (NN)')
plt.xlabel('Test R2 (NN)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path


from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_2')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

###############################
# For portfolio calculations ##
###############################

T_test = 35
rf = 0
import numpy as np
import pandas as pd

# Assuming test_pred contains the predicted returns for individual stocks
# and test_r_t contains the actual returns for the test period

def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return


def calculate_sharpe_ratio(portfolio_returns,rf):
    mean_portfolio_return = np.mean(portfolio_returns)
    std_dev_portfolio_return = np.std(portfolio_returns)
    sharpe_ratio = (mean_portfolio_return - rf ) / std_dev_portfolio_return
    return sharpe_ratio

#############################
## DATA GENERATING PROCESS ##
#############################

# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration

# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]
factors = [1, 2, 3, 4, 5, 6]
def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size]
    val_data = r_adjusted[:, train_size:train_size+val_size]
    test_data = r_adjusted[:, train_size+val_size:]

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size


###################
## calculate x_t ##
###################


# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

###################
## define CA_2 x_t ##
###################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    # First hidden layer with 32 neurons and batch normalization
    hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(input_beta)
    batch_norm_1 = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    # Second hidden layer with 16 neurons and batch normalization
    hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(batch_norm_1)
    batch_norm_2 = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(batch_norm_2)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default


#######################################
## training loop with simulated data ##
#######################################

#create columns for result scores
cols = ['l1_reg', 'factors', 'Total R^2', 'sr_xt','sr_long_short' ]

# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
results = []
for factor in tqdm(factors):
    scores = []
    for i in tqdm(range(num_simulations)):
        train_data, val_data, test_data, adjusted_R2, c_normalized, train_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

        # Prepare inputs for the factor network input calculation
        Z_t_minus_1_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size)])
        r_t_train = np.array([train_data[:, t] for t in range(1, train_size)])

        # Calculate the input for the factor network
        factor_network_input_train = calculate_factor_network_input(Z_t_minus_1_train, r_t_train)

        best_val_R2_nn = -np.inf
        best_l1_value = None

        for l1_value in l1_values:
            # Build and train the neural network model with current l1_value
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, return_components=True)

            # Prepare inputs for the model
            input_beta_train = Z_t_minus_1_train
            input_factor_train = factor_network_input_train

            # Early stopping callback
            early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

            # Train the model with early stopping
            main_model.fit(
                [input_beta_train, input_factor_train],
                train_data[:, 1:].T,
                epochs=100,  # Set a high number of epochs
                batch_size=32,
                verbose=0,
                validation_data=([input_beta_train, input_factor_train], train_data[:, 1:].T),  # Validation data is needed for early stopping
                callbacks=[early_stopping]
            )

            # Prepare validation inputs
            Z_t_minus_1_val = np.array([c_normalized[:, :, t-1] for t in range(train_size, train_size + val_data.shape[1])])
            Z_t_minus_1_val = Z_t_minus_1_val[:val_data.shape[1]]  # Ensure the shape is consistent
            val_input_beta = Z_t_minus_1_val.reshape(val_data.shape[1], N, Pc)

            val_r_t = val_data.T  # Use the returns for validation
            val_r_t = val_r_t[:val_input_beta.shape[0]]  # Ensure the shape is consistent

            # Prepare validation factor input
            val_factor_network_input = calculate_factor_network_input(val_input_beta, val_r_t)

            # Evaluate on validation data
            val_pred = main_model.predict([val_input_beta, val_factor_network_input])
            val_R2_nn = np.mean(1 - np.var(val_r_t.T - val_pred.T, axis=1) / np.var(val_r_t.T, axis=1))

            # Select the best l1_value based on validation performance
            if val_R2_nn > best_val_R2_nn:
                best_val_R2_nn = val_R2_nn
                best_l1_value = l1_value
        print('------------------------------------------------')
        print(f'Best val R^2 {best_val_R2_nn}')

        # Train final model with the best l1_value on the entire training data
        main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, return_components=True)
        main_model.fit(
            [input_beta_train, input_factor_train],
            train_data[:, 1:].T,
            epochs=100,  # Set a high number of epochs
            batch_size=32,
            verbose=0,
            validation_data=([input_beta_train, input_factor_train], train_data[:, 1:].T),  # Validation data is needed for early stopping
            callbacks=[early_stopping]
        )





        # Prepare test inputs
        Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_data.shape[1], T)])
        Z_t_minus_1_test = Z_t_minus_1_test[:test_data.shape[1]]  # Ensure the shape is consistent
        test_input_beta = Z_t_minus_1_test.reshape(test_data.shape[1], N, Pc)

        test_r_t = test_data.T  # Use the returns for testing
        test_r_t = test_r_t[:test_input_beta.shape[0]]  # Ensure the shape is consistent

        # Prepare test factor input
        test_factor_network_input = calculate_factor_network_input(test_input_beta, test_r_t)

        # Evaluate on test data
        test_pred = main_model.predict([test_input_beta, test_factor_network_input])
        test_R2_nn = np.mean(1 - np.var(test_r_t.T - test_pred.T, axis=1) / np.var(test_r_t.T, axis=1))

       #Calculate predicted x_t
        predicted_x_t = calculate_factor_network_input(test_input_beta,test_pred)
        print(f'PRECICTED XT{predicted_x_t.shape}')
        print(f'TRUE XT{test_factor_network_input.shape}')

        #Evaluate predicted x_t
        true_x_t = test_factor_network_input
        RSS = np.sum((true_x_t - predicted_x_t) ** 2)
        TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
        R2_x_t = 1 - (RSS / TSS)
        print(f'R^2 for predicted x_t: {R2_x_t}')

        # Calculate the Sharpe ratio for predicted x_t OOS
        # risk_free_rate = 0  # Define the risk-free rate
        # mean_return = np.mean(predicted_x_t)
        # std_dev_return = np.std(predicted_x_t)
        # sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return

        sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t,rf)
        print('------------------------------------------------')
        print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
        print('------------------------------------------------')

        # Calculate the portfolio return for each time period
        portfolio_returns = np.zeros(T_test)
        for t in range(T_test):
            portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], test_r_t[:, t])

        # Calculate the cumulative portfolio return
        cumulative_portfolio_return = np.cumsum(portfolio_returns)

        # Create a DataFrame for the portfolio returns
        start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
        date_range = pd.date_range(start_date, periods=T_test, freq='M')
        df_portfolio_returns = pd.DataFrame({
            'Date': date_range,
            'Portfolio_Return': portfolio_returns,
            'Cumulative_Portfolio_Return': cumulative_portfolio_return
        })

        long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns,rf)

        print('------------------------------------------------')
        print(f'Sharpe ratio for long-short portfolio reutrns :{long_short_portfolio_sharpe_ratio}')

        print('------------------------------------------------')
        print(f'Iteration : {i}|TEST R^2 :{test_R2_nn}')
        results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
        print('------------------------------------------------')
        if (i + 1) % 10 == 0:
            print(f'Simulation {i + 1}/{num_simulations} completed')

        scores.append([best_l1_value, factors, test_R2_nn,sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio ])

    scores_df = pd.DataFrame(scores, columns=cols)
    scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{best_l1_value}/{factors}', append=True, format='table')


    # Aggregate Results
    adjusted_R2_values = [res[0] for res in results]
    val_R2_nn_values = [res[1] for res in results]
    test_R2_nn_values = [res[2] for res in results]

    mean_adjusted_R2 = np.mean(adjusted_R2_values)
    std_adjusted_R2 = np.std(adjusted_R2_values)

    mean_val_R2_nn = np.mean(val_R2_nn_values)
    std_val_R2_nn = np.std(val_R2_nn_values)

    mean_test_R2_nn = np.mean(test_R2_nn_values)
    std_test_R2_nn = np.std(test_R2_nn_values)

    print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
    print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

    print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
    print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

    print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
    print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

    # Plotting the distribution of Adjusted R2
    plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
    plt.title('Distribution of Adjusted R2')
    plt.xlabel('Adjusted R2')
    plt.ylabel('Frequency')
    plt.show()

    # Plotting the distribution of Validation R2 (NN)
    plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
    plt.title('Distribution of Validation R2 (NN)')
    plt.xlabel('Validation R2 (NN)')
    plt.ylabel('Frequency')
    plt.show()

    # Plotting the distribution of Test R2 (NN)
    plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
    plt.title('Distribution of Test R2 (NN)')
    plt.xlabel('Test R2 (NN)')
    plt.ylabel('Frequency')
    plt.show()


**FOR ALL MODELS AND ALL ITERATIONS FOR EACH FACTOR - FORWARD RETURNS**

In [ ]:
!pip install tensorflow-gpu


In [ ]:
import tensorflow as tf
print("Num GPUs Available: ", len(tf.config.experimental.list_physical_devices('GPU')))


**forward returns predicted**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path



from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_PRED')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)



# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration


####################################
## DEFINE DATA GENERATING PROCESS ##
####################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)

    train_data = r_adjusted[:, :train_size+1]  # Include the next period for forward returns :  train_data = r_adjusted[:, :train_size+1]
    val_data = r_adjusted[:, train_size:train_size+val_size+1]  # Include the next period for forward returns :  val_data = r_adjusted[:, train_size:train_size+val_size+1]
    test_data = r_adjusted[:, train_size+val_size:T]  # Last segment, no need for forward returns here

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size



###############################################################################
## Define CA model with a dynamic number of layers that can be iterated over ##
###############################################################################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    x = input_beta

    if num_hidden_layers >= 1:
        # First hidden layer with 32 neurons and batch normalization
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    if num_hidden_layers >= 2:
        # Second hidden layer with 16 neurons and batch normalization
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    if num_hidden_layers >= 3:
        # Third hidden layer with 8 neurons and batch normalization
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

###########################################################################
## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
###########################################################################

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

# Function for calculation of the sharpe ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

# Function to calculate the long-short portfolio
def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return

#######################################
## DEFINE PARAMETERS TO ITERATE OVER ##
#######################################

# Define the different model configurations
models = [0, 1, 2, 3]  # Number of hidden layers

# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
factors = [1, 2, 3, 4, 5]  # Example factor list
#
# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]
#

cols = ['Model','Factors' ,'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio','Test R^2','Best L1 Value']

#######################################
## training loop with simulated data ##
#######################################

results = []  # Define results here

for num_hidden_layers in tqdm(models):
    for factor in tqdm(factors):
        scores = []
        for i in tqdm(range(num_simulations)):
            train_data, val_data, test_data, adjusted_R2, c_normalized, train_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

            # Prepare inputs for the factor network input calculation
            Z_t_minus_1_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size)])
            r_t_train = np.array([train_data[:, t+1] for t in range(train_size - 1)])  # Use forward returns : r_t_train = np.array([train_data[:, t+1] for t in range(train_size - 1)])

            # Calculate the input for the factor network
            factor_network_input_train = calculate_factor_network_input(Z_t_minus_1_train, r_t_train)

            best_val_R2_nn = -np.inf
            best_l1_value = None

            for l1_value in l1_values:
                # Build and train the neural network model with current l1_value
                main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)

                # Prepare inputs for the model
                input_beta_train = Z_t_minus_1_train
                input_factor_train = factor_network_input_train

                # Early stopping callback
                early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


                # Prepare validation inputs
                Z_t_minus_1_val = np.array([c_normalized[:, :, t-1] for t in range(train_size, train_size + val_data.shape[1] - 1)])
                val_input_beta = Z_t_minus_1_val.reshape(val_data.shape[1] - 1, N, Pc)

                val_r_t = val_data[:, 1:].T  # Use the forward returns for validation => no forward returns : val_r_t = val_data.T

                # Ensure the shape is consistent
                assert val_input_beta.shape[0] == val_r_t.shape[0], "Shape mismatch in validation data"

                # Prepare validation factor input
                val_factor_network_input = calculate_factor_network_input(val_input_beta, val_r_t)

                # Train the model with early stopping
                main_model.fit(
                    [input_beta_train, input_factor_train],
                    train_data[:, 1:train_size].T,  # Use appropriate train data slice
                    epochs=100,  # Set a high number of epochs
                    batch_size=45,
                    verbose=0,
                    validation_data=([val_input_beta, val_factor_network_input], val_data[:, 1:train_size].T),  # Validation data is needed for early stopping
                    callbacks=[early_stopping]
                )


                # Evaluate on validation data
                val_pred = main_model.predict([val_input_beta, val_factor_network_input])
                val_R2_nn = np.mean(1 - np.var(val_r_t - val_pred, axis=1) / np.var(val_r_t, axis=1))

                # Select the best l1_value based on validation performance
                if val_R2_nn > best_val_R2_nn:
                    best_val_R2_nn = val_R2_nn
                    best_l1_value = l1_value
            print('------------------------------------------------')
            print(f'Best val R^2 {best_val_R2_nn}')

            # Train final model with the best l1_value on the entire training data
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)
            main_model.fit(
                [input_beta_train, input_factor_train],
                train_data[:, 1:train_size].T,  # Use appropriate train data slice
                epochs=100,  # Set a high number of epochs
                batch_size=45,
                verbose=0,
                validation_data=([input_beta_train, input_factor_train], train_data[:, 1:train_size].T),  # Validation data is needed for early stopping
                callbacks=[early_stopping]
            )

            # Prepare test inputs
            Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_data.shape[1], T - 1)])
            test_input_beta = Z_t_minus_1_test.reshape(test_data.shape[1] - 1, N, Pc)

            test_r_t = test_data[:, 1:].T  # Use the forward returns for testing : test_r_t = test_data[:, 1:].T

            # no forward returns : test_r_t = test_data.T

            # Ensure the shape is consistent
            assert test_input_beta.shape[0] == test_r_t.shape[0], "Shape mismatch in test data"

            # Prepare test factor input
            test_factor_network_input = calculate_factor_network_input(test_input_beta, test_r_t)

            # Evaluate on test data
            test_pred = main_model.predict([test_input_beta, test_factor_network_input])
            test_R2_nn = np.mean(1 - np.var(test_r_t - test_pred, axis=1) / np.var(test_r_t, axis=1))
            print(f'TEST R^2 : {test_R2_nn}')
            # Calculate predicted x_t
            predicted_x_t = calculate_factor_network_input(test_input_beta, test_pred)
            # print(f'PREDICTED XT {predicted_x_t.shape}')
            # print(f'TRUE XT {test_factor_network_input.shape}')

            # Evaluate predicted x_t
            true_x_t = test_factor_network_input
            RSS = np.sum((true_x_t - predicted_x_t) ** 2)
            TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
            R2_x_t = 1 - (RSS / TSS)
            print(f'R^2 for TEST x_t: {R2_x_t}')

            # Calculate the Sharpe ratio for predicted x_t OOS
            sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)
            print('------------------------------------------------')
            print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
            print('------------------------------------------------')

            # Calculate the portfolio return for each time period
            portfolio_returns = np.zeros(test_r_t.shape[1])
            for t in range(test_r_t.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], test_r_t[:, t])

            # Calculate the cumulative portfolio return
            cumulative_portfolio_return = np.cumsum(portfolio_returns)

            # Create a DataFrame for the portfolio returns
            start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
            date_range = pd.date_range(start_date, periods=test_r_t.shape[1], freq='M')
            df_portfolio_returns = pd.DataFrame({
                'Date': date_range,
                'Portfolio_Return': portfolio_returns,
                'Cumulative_Portfolio_Return': cumulative_portfolio_return
            })

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            print('------------------------------------------------')
            print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

            print('------------------------------------------------')
            print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
            results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
            print('------------------------------------------------')
            if (i + 1) % 10 == 0:
                print(f'Simulation {i + 1}/{num_simulations} completed')

            scores.append([str(num_hidden_layers) ,factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value ])

        # Convert scores list to DataFrame
        scores_df = pd.DataFrame(scores, columns=cols)

        # Save scores_df to HDF5 file for the current factor
        scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{num_hidden_layers}_hidden_layers/{factor}', append=True, format='table')

# Aggregate Results
adjusted_R2_values = [res[0] for res in results]
val_R2_nn_values = [res[1] for res in results]
test_R2_nn_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2_nn = np.mean(val_R2_nn_values)
std_val_R2_nn = np.std(val_R2_nn_values)

mean_test_R2_nn = np.mean(test_R2_nn_values)
std_test_R2_nn = np.std(test_R2_nn_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

# Plotting the distribution of Adjusted R2
plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Validation R2 (NN)
plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2 (NN)')
plt.xlabel('Validation R2 (NN)')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Test R2 (NN)
plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2 (NN)')
plt.xlabel('Test R2 (NN)')
plt.ylabel('Frequency')
plt.show()


**Contemporary Returns Predicted**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path



from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_PRED')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)



# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration


####################################
## DEFINE DATA GENERATING PROCESS ##
####################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = int(T*(1 - (val_ratio + train_ratio)))

    train_data = r_adjusted[:, :train_size]  # Include the next period for forward returns :  train_data = r_adjusted[:, :train_size+1]
    val_data = r_adjusted[:, train_size:train_size+val_size]  # Include the next period for forward returns :  val_data = r_adjusted[:, train_size:train_size+val_size+1]
    test_data = r_adjusted[:, test_size:T]  # Last segment, no need for forward returns here

    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size



###############################################################################
## Define CA model with a dynamic number of layers that can be iterated over ##
###############################################################################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    x = input_beta

    if num_hidden_layers >= 1:
        # First hidden layer with 32 neurons and batch normalization
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    if num_hidden_layers >= 2:
        # Second hidden layer with 16 neurons and batch normalization
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    if num_hidden_layers >= 3:
        # Third hidden layer with 8 neurons and batch normalization
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

###########################################################################
## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
###########################################################################

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

# Function for calculation of the sharpe ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

# Function to calculate the long-short portfolio
def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return

#######################################
## DEFINE PARAMETERS TO ITERATE OVER ##
#######################################

# Define the different model configurations
models = [0, 1, 2, 3]  # Number of hidden layers

# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
factors = [1, 2, 3, 4, 5]  # Example factor list
#
# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]
#

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


cols = ['Model','Factors' ,'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio','Test R^2','Best L1 Value', 'R^2 DGP']

#######################################
## training loop with simulated data ##
#######################################

results = []  # Define results here

for num_hidden_layers in tqdm(models):
    for factor in tqdm(factors):
        scores = []
        for i in tqdm(range(num_simulations)):
            train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

            ################
            ## TRAIN DATA ##
            ################

            # Prepare inputs for the factor network input calculation
            input_beta_train = np.array([c_normalized[:, :, t-1] for t in range(train_size)])
            r_t_train = np.array([train_data[:, t] for t in range(train_size)])

            # Calculate the input for the factor network
            input_factor_train = calculate_factor_network_input(input_beta_train, r_t_train)

            # Ensure the shape is consistent
            assert input_beta_train.shape[0] == r_t_train.shape[0], "Shape mismatch in validation data"

            ######################
            ### VALIDATION DATA ##
            ######################

            input_beta_val = np.array([c_normalized[:, :, t-1] for t in range(val_size)])  # Adjusted range
            r_t_val = np.array([val_data[:, t] for t in range(val_size)])

            # Calculate the input for the factor network
            input_factor_val = calculate_factor_network_input(input_beta_val, r_t_val)

            # Ensure the shape is consistent
            assert input_beta_val.shape[0] == r_t_val.shape[0], "Shape mismatch in validation data"


            #################
            ### TEAST DATA ##
            #################
            input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(test_size)])  # Adjusted range
            r_t_test = np.array([test_data[:, t] for t in range(test_size)])

            # Calculate the input for the factor network
            input_factor_test = calculate_factor_network_input(input_beta_test, r_t_test)


            # Ensure the shape is consistent
            assert input_beta_test.shape[0] == r_t_test.shape[0], "Shape mismatch in test data"

            best_val_R2_nn = -np.inf
            best_l1_value = None

            for l1_value in l1_values:
                # Build and train the neural network model with current l1_value
                main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)


                # Train the model with early stopping
                main_model.fit(
                    [input_beta_train, input_factor_train],
                    r_t_train,  # Use appropriate train data slice
                    epochs=100,  # Set a high number of epochs
                    batch_size=45,
                    verbose=0,
                    validation_data=([input_beta_val, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                    callbacks=[early_stopping]
                )

                # Evaluate on validation data
                val_pred = main_model.predict([input_beta_val, input_factor_val])
                val_R2_nn = np.mean(1 - np.var(r_t_val - val_pred, axis=1) / np.var(r_t_val, axis=1))

                # Select the best l1_value based on validation performance
                if val_R2_nn > best_val_R2_nn:
                    best_val_R2_nn = val_R2_nn
                    best_l1_value = l1_value
            print('------------------------------------------------')
            print(f'Best val R^2 {best_val_R2_nn}')

            # Train final model with the best l1_value on the entire training data
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)
            main_model.fit(
                [input_beta_train, input_factor_train],
                r_t_train,  # Use appropriate train data slice
                epochs=100,  # Set a high number of epochs
                batch_size=45,
                verbose=0,
                validation_data=([input_beta_val, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                callbacks=[early_stopping]
            )


            # Evaluate on test data
            test_pred = main_model.predict([input_beta_test, input_factor_test])

            test_R2_nn = np.mean(1 - np.var(r_t_test - test_pred, axis=1) / np.var(r_t_test, axis=1))

            print(f'TEST R^2 : {test_R2_nn}')
            # Calculate predicted x_t
            predicted_x_t = calculate_factor_network_input(input_beta_test, test_pred)
            # print(f'PREDICTED XT {predicted_x_t.shape}')
            # print(f'TRUE XT {test_factor_network_input.shape}')

            # Evaluate predicted x_t
            true_x_t = input_factor_test
            RSS = np.sum((true_x_t - predicted_x_t) ** 2)
            TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
            R2_x_t = 1 - (RSS / TSS)
            print(f'R^2 for TEST x_t: {R2_x_t}')

            # Calculate the Sharpe ratio for predicted x_t OOS
            sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)
            print('------------------------------------------------')
            print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
            print('------------------------------------------------')

            # Calculate the portfolio return for each time period
            portfolio_returns = np.zeros(r_t_test.shape[1])
            for t in range(r_t_test.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], r_t_test[:, t])

            # Calculate the cumulative portfolio return
            cumulative_portfolio_return = np.cumsum(portfolio_returns)

            # Create a DataFrame for the portfolio returns
            start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
            date_range = pd.date_range(start_date, periods=r_t_test.shape[1], freq='M')
            df_portfolio_returns = pd.DataFrame({
                'Date': date_range,
                'Portfolio_Return': portfolio_returns,
                'Cumulative_Portfolio_Return': cumulative_portfolio_return
            })

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            print('------------------------------------------------')
            print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

            print('------------------------------------------------')
            print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
            results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
            print('------------------------------------------------')
            if (i + 1) % 10 == 0:
                print(f'Simulation {i + 1}/{num_simulations} completed')

            scores.append([str(num_hidden_layers) ,factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value, adjusted_R2])

        # Convert scores list to DataFrame
        scores_df = pd.DataFrame(scores, columns=cols)

        # Save scores_df to HDF5 file for the current factor
        scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{num_hidden_layers}_hidden_layers/{factor}', append=True, format='table')

# Aggregate Results
adjusted_R2_values = [res[0] for res in results]
val_R2_nn_values = [res[1] for res in results]
test_R2_nn_values = [res[2] for res in results]

mean_adjusted_R2 = np.mean(adjusted_R2_values)
std_adjusted_R2 = np.std(adjusted_R2_values)

mean_val_R2_nn = np.mean(val_R2_nn_values)
std_val_R2_nn = np.std(val_R2_nn_values)

mean_test_R2_nn = np.mean(test_R2_nn_values)
std_test_R2_nn = np.std(test_R2_nn_values)

print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

# Plotting the distribution of Adjusted R2
plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
plt.title('Distribution of Adjusted R2')
plt.xlabel('Adjusted R2')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Validation R2 (NN)
plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Validation R2 (NN)')
plt.xlabel('Validation R2 (NN)')
plt.ylabel('Frequency')
plt.show()

# Plotting the distribution of Test R2 (NN)
plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
plt.title('Distribution of Test R2 (NN)')
plt.xlabel('Test R2 (NN)')
plt.ylabel('Frequency')
plt.show()


**Linear model $\rightarrow$ OOS TOTAL R^2**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path

!pip install tables


##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################
from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_TOTAL_LINEAR_x_t')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

#####################################################################################################
## IF RUNTIME DISCONNECTS, THEN LOAD SAVED SCORES AND KEEP ON TRAININ FROM WHERE THE CODE LEFT OFF ##
#####################################################################################################

# # Load the saved scores and determine completed configurations
# def load_scores(results_path, models, factors):
#     completed_configs = set()
#     for num_hidden_layers in models:
#         for factor in factors:
#             file_path = results_path / f'scores_{num_hidden_layers}_{factor}.h5'
#             if file_path.exists():
#                 completed_configs.add((num_hidden_layers, factor))
#     return completed_configs





# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration


####################################
## DEFINE DATA GENERATING PROCESS ##
####################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
                                         c_normalized[:, 1, t],
                                         0.8 * c_normalized[:, 2, t],
                                         0.6 * c_normalized[:, 3, t],
                                         0.4 * c_normalized[:, 4, t]]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    # train_ratio = 0.6
    # val_ratio = 0.2
    # train_size = int(T * train_ratio)
    # val_size = int(T * val_ratio)
    # test_size = int(T*(1 - train_ratio - val_ratio)-1) # subtract the last time period because the characteristics are lagged

    # train_data = r_adjusted[:, :train_size]  # Training data up to train_size
    # val_data = r_adjusted[:, train_size:train_size+val_size]  # Validation data from train_size to train_size + val_size
    # test_data = r_adjusted[:, train_size+val_size:]  # Testing data from train_size + val_size to the end

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1  # subtract the last time period because the characteristics are lagged test size defines the size of the index of the returns.

    train_data = r_adjusted[:, :train_size]  # Training data up to train_size
    val_data = r_adjusted[:, train_size:train_size + val_size]  # Validation data from train_size to train_size + val_size
    test_data = r_adjusted[:, train_size + val_size :]  # Testing data from train_size + val_size to the end


    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size



###############################################################################
## Define CA model with a dynamic number of layers that can be iterated over ##
###############################################################################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    x = input_beta

    if num_hidden_layers >= 1:
        # First hidden layer with 32 neurons and batch normalization
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    if num_hidden_layers >= 2:
        # Second hidden layer with 16 neurons and batch normalization
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    if num_hidden_layers >= 3:
        # Third hidden layer with 8 neurons and batch normalization
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

###########################################################################
## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
###########################################################################

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

# Function for calculation of the sharpe ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

# Function to calculate the long-short portfolio
def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return


#######################################
## DEFINE PARAMETERS TO ITERATE OVER ##
#######################################

# Define the different model configurations
models = [0,1,2,3]  # Number of hidden layers
#
# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
factors = [1,2, 3, 4, 5, 6]  # Example factor list
#,
# L1 regularization values to search over
l1_values = [0.00001, 0.001]
#

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# Define columns
cols = ['Model', 'Factors', '$x_t$ Sharpe Ratio', 'Long-Short Portfolio Sharpe Ratio', '$R^2$ for $r_t$', 'Best L_1 Value', 'Adjusted $R^2$', '$R^2$ for $x_t$' ]

results = []  # Define results here
scores_dict = {} # Define scores_dict here
#######################################
## training loop with simulated data ##
#######################################

results = []  # Define results here

for num_hidden_layers in tqdm(models):
    for factor in tqdm(factors):
        scores = []
        for i in tqdm(range(num_simulations)):
            train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)


            print(f'Train data{train_data.shape}')
            print(f'Val data shape :{val_data.shape}')
            print(f'Test data shape :{test_data.shape}')

            print(f'Train size:{train_size}')
            print(f'Val size :{val_size}')
            print(f'Test size :{test_size}')
            ################
            ## TRAIN DATA ##
            ################

            # Prepare inputs for the factor network input calculation
            input_beta_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size + 1)])

            r_t_train = np.array([train_data[:, t] for t in range(train_size)])


            print(input_beta_train.shape, r_t_train.shape)
            # Calculate the input for the factor network
            input_factor_train = calculate_factor_network_input(input_beta_train, r_t_train)

            # Ensure the shape is consistent
            assert input_beta_train.shape[0] == r_t_train.shape[0], "Shape mismatch in validation data"

            ######################
            ### VALIDATION DATA ##
            ######################

            input_beta_val = np.array([c_normalized[:, :, t-1] for t in range(train_size + 1, train_size + val_size + 1)])
            r_t_val = np.array([val_data[:, t] for t in range(val_size)])

            print(f'beta input VAL{input_beta_val.shape}')
            print(f'r_t VAL{r_t_val.shape}')

            # Calculate the input for the factor network
            input_factor_val = calculate_factor_network_input(input_beta_val, r_t_val)

            print(f'beta input VAL {input_beta_val.shape}')
            print(f'r_t VAL {r_t_val.shape}')

            # Ensure the shape is consistent
            assert input_beta_val.shape[0] == r_t_val.shape[0], "Shape mismatch in validation data"


            #################
            ### TEAST DATA ##
            #################


            # Ensure that the input_beta_test and r_t_test ranges are correctly aligned
            input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])

            # # Ensure that the input_beta_test and r_t_test ranges are correctly aligned
            # input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            # r_t_test = np.array([test_data[:, t ] for t in range(test_size)])

            print(f'beta input TEST {input_beta_test.shape}')

            # Print shapes for debugging
            print("Test shapes:", input_beta_test.shape, r_t_test.shape)

            # Calculate the input for the factor network
            input_factor_test = calculate_factor_network_input(input_beta_test, r_t_test)


            # Ensure the shape is consistent
            assert input_beta_test.shape[0] == r_t_test.shape[0], "Shape mismatch in test data"

            best_val_R2_nn = -np.inf
            best_l1_value = None

            for l1_value in l1_values:
                # Build and train the neural network model with current l1_value
                main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)


                # Train the model with early stopping
                main_model.fit(
                    [input_beta_train, input_factor_train],
                    r_t_train,  # Use appropriate train data slice
                    epochs=100,  # Set a high number of epochs
                    batch_size=45,
                    verbose=0,
                    validation_data=([input_beta_val, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                    callbacks=[early_stopping]
                )

                # Evaluate on validation data
                val_pred = main_model.predict([input_beta_val, input_factor_val])
                val_R2_nn = np.mean(1 - np.var(r_t_val - val_pred, axis=1) / np.var(r_t_val, axis=1))

                # Select the best l1_value based on validation performance
                if val_R2_nn > best_val_R2_nn:
                    best_val_R2_nn = val_R2_nn
                    best_l1_value = l1_value
            print('------------------------------------------------')
            print(f'Best val R^2 {best_val_R2_nn}')

            # Train final model with the best l1_value on the entire training data
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)
            main_model.fit(
                [input_beta_train, input_factor_train],
                r_t_train,  # Use appropriate train data slice
                epochs=100,  # Set a high number of epochs
                batch_size=45,
                verbose=0,
                validation_data=([input_beta_val, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                callbacks=[early_stopping]
            )


            # Evaluate on test data
            test_pred = main_model.predict([input_beta_test, input_factor_test])

            test_R2_nn = np.mean(1 - np.var(r_t_test - test_pred, axis=1) / np.var(r_t_test, axis=1))

            print(f'TEST R^2 : {test_R2_nn}')
            # Calculate predicted x_t
            predicted_x_t = calculate_factor_network_input(input_beta_test, test_pred)
            # print(f'PREDICTED XT {predicted_x_t.shape}')
            # print(f'TRUE XT {test_factor_network_input.shape}')

            # Evaluate predicted x_t
            true_x_t = input_factor_test
            RSS = np.sum((true_x_t - predicted_x_t) ** 2)
            TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
            R2_x_t = 1 - (RSS / TSS)
            print(f'R^2 for TEST x_t: {R2_x_t}')

            # Calculate the Sharpe ratio for predicted x_t OOS
            sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)
            print('------------------------------------------------')
            print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
            print('------------------------------------------------')

            # Calculate the portfolio return for each time period
            portfolio_returns = np.zeros(r_t_test.shape[1])
            for t in range(r_t_test.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], r_t_test[:, t])

            # Calculate the cumulative portfolio return
            cumulative_portfolio_return = np.cumsum(portfolio_returns)

            # Create a DataFrame for the portfolio returns
            start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
            date_range = pd.date_range(start_date, periods=r_t_test.shape[1], freq='M')
            df_portfolio_returns = pd.DataFrame({
                'Date': date_range,
                'Portfolio_Return': portfolio_returns,
                'Cumulative_Portfolio_Return': cumulative_portfolio_return
            })

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            print('------------------------------------------------')
            print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

            print('------------------------------------------------')
            print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
            results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
            print('------------------------------------------------')
            if (i + 1) % 10 == 0:
                print(f'Simulation {i + 1}/{num_simulations} completed')
            scores.append([str(num_hidden_layers) ,factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value, adjusted_R2, R2_x_t])


        # Convert scores list to DataFrame
        scores_df = pd.DataFrame(scores, columns=cols)
        scores_dict[(num_hidden_layers, factor)] = scores_df

        scores_df.to_hdf(str(results_path / f'scores_{num_hidden_layers}_{factor}.h5'), key='scores', mode='w')



# Plotting the results for each model and factor
for (num_hidden_layers, factor), df in scores_dict.items():
    plt.figure(figsize=(20, 5))  # Adjusted figure size to accommodate the extra subplot

    plt.subplot(1, 4, 1)
    plt.hist(df['Adjusted $R^2$'], bins=20, edgecolor='black')
    plt.title(f'Adjusted $R^2$ for $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Adjusted $R^2$')
    plt.ylabel('Frequency')

    plt.subplot(1, 4, 2)
    plt.hist(df['$R^2$ for $r_t$'], bins=20, edgecolor='black')
    plt.title(f'Test $R^2$ (NN) for $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('$R^2$ for $r_t$')
    plt.ylabel('Frequency')

    plt.subplot(1, 4, 3)
    plt.hist(df['$x_t$ Sharpe Ratio'], bins=20, edgecolor='black')
    plt.title(f'Sharpe Ratio $x_t$ for Model $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Sharpe Ratio $x_t$')
    plt.ylabel('Frequency')

    plt.subplot(1, 4, 4)  # Added subplot for the long-short portfolio Sharpe ratio
    plt.hist(df['Long-Short Portfolio Sharpe Ratio'], bins=20, edgecolor='black')
    plt.title(f'Long-Short Portfolio Sharpe Ratio for Model $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Long-Short Portfolio Sharpe Ratio')
    plt.ylabel('Frequency')

    plt.tight_layout()

    # Save the plot
    plot_filename = results_path / f'model_{num_hidden_layers}_factor_{factor}.png'
    plt.savefig(plot_filename)
    plt.close()



In [ ]:
import pandas as pd
from pathlib import Path

!pip install tables
from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_TOTAL_LINEAR_x_t')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

def load_scores(results_path, models, factors):
    completed_configs = set()
    scores_dict = {}

    for num_hidden_layers in models:
        for factor in factors:
            file_path = results_path / f'scores_{num_hidden_layers}_{factor}.h5'
            if file_path.exists():
                completed_configs.add((num_hidden_layers, factor))
                scores_df = pd.read_hdf(file_path, key='scores')
                scores_dict[(num_hidden_layers, factor)] = scores_df

    return completed_configs, scores_dict

# Example usage:
models = [0]  # Define your model configurations
factors = [1]  # Define your factor configurations

completed_configs, scores_dict = load_scores(results_path, models, factors)

# Now you can access the loaded scores from the `scores_dict`
for (num_hidden_layers, factor), df in scores_dict.items():
    print(f'Model: {num_hidden_layers}, Factor: {factor}')
    print(df)


In [ ]:

from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_TOTAL_x_t')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

# Load the saved scores and determine completed configurations
def load_scores(results_path, models, factors):
    completed_configs = set()
    for num_hidden_layers in models:
        for factor in factors:
            file_path = results_path / f'scores_{num_hidden_layers}_{factor}.h5'
            if file_path.exists():
                completed_configs.add((num_hidden_layers, factor))
    return completed_configs


# Determine completed configurations
completed_configs = load_scores(results_path, models, factors)

print(completed_configs)

**Linear model $\rightarrow$ Predictive R^2**

here, we are using the lagged factor loadings and the lagged factors to predict the returns.

In [ ]:
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# from sklearn.decomposition import PCA
# from tqdm import tqdm
# from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
# from tensorflow.keras.models import Model
# from tensorflow.keras.regularizers import l1
# from tensorflow.keras.callbacks import EarlyStopping
# from pathlib import Path



# # from google.colab import drive
# # drive.mount('/content/drive')

# # # Define the base path for Google Drive
# # base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_PRED')

# # # Define the results path relative to the base path
# # results_path = base_path / 'results' / 'asset_pricing'

# # # Check if the path exists, if not, create it
# # if not results_path.exists():
# #     results_path.mkdir(parents=True)



# # Define parameters
# N = 200  # Number of assets
# T = 180  # Time periods
# Pc = 50  # Number of characteristics
# Px = 50  # Number of factor components
# num_factors = 5  # Number of factors
# R2_target = 0.45  # Target R2 for calibration


# ####################################
# ## DEFINE DATA GENERATING PROCESS ##
# ####################################

# def generate_data(N, T, Pc, Px, num_factors, R2_target):
#     # Simulate characteristics
#     rho = np.random.uniform(0.9, 1, Pc)
#     c = np.zeros((N, Pc, T))
#     for t in range(1, T):
#         c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

#     # Normalize characteristics
#     c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

#     # Generate factors and errors
#     x = np.random.normal(0.03, 0.12, (Px, T))
#     eta = np.random.normal(0, 0.01, (num_factors, T))
#     epsilon = np.random.standard_t(5, (N, T))

#     # Define linear factor loadings
#     beta_linear = np.zeros((N, num_factors, T))
#     for t in range(1, T):
#         beta_linear[:, :, t] = np.array([1.2 * c_normalized[:, 0, t],
#                                          c_normalized[:, 1, t],
#                                          0.8 * c_normalized[:, 2, t],
#                                          0.6 * c_normalized[:, 3, t],
#                                          0.4 * c_normalized[:, 4, t]]).T

#     # Initial Returns Calculation
#     f = np.dot(np.random.randn(num_factors, Px), x) + eta
#     r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

#     # Calculate initial R2
#     explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
#     total_variance = np.var(r, axis=1)
#     initial_R2 = np.mean(explained_variance / total_variance)

#     # Adjust the variances to achieve the target R2
#     adjustment_factor = np.sqrt(R2_target / initial_R2)
#     f_adjusted = adjustment_factor * f
#     epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

#     # Generate adjusted returns
#     r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

#     # Calculate adjusted R2
#     explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
#     total_variance_adjusted = np.var(r_adjusted, axis=1)
#     adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

#     # Split data into training, validation, and testing sets
#     # train_ratio = 0.6
#     # val_ratio = 0.2
#     # train_size = int(T * train_ratio)
#     # val_size = int(T * val_ratio)
#     # test_size = int(T*(1 - train_ratio - val_ratio)-1) # subtract the last time period because the characteristics are lagged

#     # train_data = r_adjusted[:, :train_size]  # Training data up to train_size
#     # val_data = r_adjusted[:, train_size:train_size+val_size]  # Validation data from train_size to train_size + val_size
#     # test_data = r_adjusted[:, train_size+val_size:]  # Testing data from train_size + val_size to the end

#     # Split data into training, validation, and testing sets
#     train_ratio = 0.6
#     val_ratio = 0.2
#     train_size = int(T * train_ratio)
#     val_size = int(T * val_ratio)
#     test_size = T - train_size - val_size - 1  # subtract the last time period because the characteristics are lagged test size defines the size of the index of the returns.

#     train_data = r_adjusted[:, :train_size]  # Training data up to train_size
#     val_data = r_adjusted[:, train_size:train_size + val_size]  # Validation data from train_size to train_size + val_size
#     test_data = r_adjusted[:, train_size + val_size :]  # Testing data from train_size + val_size to the end


#     return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size



# ###############################################################################
# ## Define CA model with a dynamic number of layers that can be iterated over ##
# ###############################################################################


# def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
#     input_beta = Input((N, Pc), name='input_beta')
#     input_factor = Input((Pc,), name='input_factor')

#     x = input_beta

#     if num_hidden_layers >= 1:
#         # First hidden layer with 32 neurons and batch normalization
#         hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
#         x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

#     if num_hidden_layers >= 2:
#         # Second hidden layer with 16 neurons and batch normalization
#         hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
#         x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

#     if num_hidden_layers >= 3:
#         # Third hidden layer with 8 neurons and batch normalization
#         hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
#         x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

#     # Output beta layer with L1 regularization
#     output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

#     # Output factor layer with L1 regularization
#     output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

#     # Dot product layer combining output_beta and output_factor
#     output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

#     # Compile the main model
#     model = Model(inputs=[input_beta, input_factor], outputs=output)
#     model.compile(loss='mse', optimizer='adam')

#     # Create a standalone factor network model
#     factor_model = Model(inputs=input_factor, outputs=output_factor)

#     if return_components:
#         return model, factor_model  # Return both the combined model and the factor network
#     else:
#         return model  # Return only the combined model by default

# ###########################################################################
# ## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
# ###########################################################################

# # Function to calculate the input for the factor network
# def calculate_factor_network_input(Z_t_minus_1, r_t):
#     factor_input = []
#     for t in range(Z_t_minus_1.shape[0]):
#         Z_t = Z_t_minus_1[t]
#         r_t_i = r_t[t]
#         Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
#         factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
#         factor_input.append(factor_input_t)
#     return np.array(factor_input)

# # Function for calculation of the sharpe ratio
# def calculate_sharpe_ratio(returns, risk_free_rate=0):
#     mean_return = np.mean(returns)
#     std_dev_return = np.std(returns)
#     sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
#     return sharpe_ratio

# # Function to calculate the long-short portfolio
# def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
#     # Number of assets
#     N = predicted_returns.shape[0]

#     # Determine the threshold for the top and bottom deciles
#     top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
#     bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

#     # Identify the long and short positions
#     long_positions = predicted_returns >= top_decile_threshold
#     short_positions = predicted_returns <= bottom_decile_threshold

#     # Calculate the returns for the long and short positions
#     long_returns = actual_returns[long_positions].mean(axis=0)
#     short_returns = actual_returns[short_positions].mean(axis=0)

#     # Calculate the portfolio return (long returns - short returns)
#     portfolio_return = long_returns - short_returns

#     return portfolio_return

# #######################################
# ## DEFINE PARAMETERS TO ITERATE OVER ##
# #######################################

# # Define the different model configurations
# models = [0, 1, 2, 3]  # Number of hidden layers

# # Monte Carlo Simulation with L1 tuning and Early Stopping
# num_simulations = 100
# factors = [1, 2, 3, 4, 5]  # Example factor list
# #
# # L1 regularization values to search over
# l1_values = [0.0001, 0.001, 0.01, 0.1]
# #

# # Early stopping callback
# early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# cols = ['Model','Factors' ,'Sharpe Ratio x_t', 'Long-Short Portfolio Sharpe Ratio','Test R^2','Best L1 Value', 'R^2 DGP']

# #######################################
# ## training loop with simulated data ##
# #######################################


# # Adjusted training loop with correctly aligned inputs
# results = []  # Define results here

# for num_hidden_layers in tqdm(models):
#     for factor in tqdm(factors):
#         scores = []
#         for i in tqdm(range(num_simulations)):
#             train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)

#             print(f'Train data {train_data.shape}')
#             print(f'Val data shape: {val_data.shape}')
#             print(f'Test data shape: {test_data.shape}')

#             print(f'Train size: {train_size}')
#             print(f'Val size: {val_size}')
#             print(f'Test size: {test_size}')

#             ################
#             ## TRAIN DATA ##
#             ################

#             # Prepare inputs for the factor network input calculation
#             input_beta_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size + 1)])

#             r_t_train = np.array([train_data[:, t] for t in range(train_size)])


#             # # Prepare inputs for the factor network input calculation
#             # input_beta_train = np.array([c_normalized[:, :, t] for t in range(train_size)])[:-1]
#             # r_t_train = np.array([train_data[:, t] for t in range(train_size)])

#             # Ensure the shape is consistent
#             assert input_beta_train.shape[0] == r_t_train.shape[0], "Shape mismatch in train data"

#             # Calculate the input for the factor network
#             input_factor_train = calculate_factor_network_input(input_beta_train, r_t_train)

#             ######################
#             ### VALIDATION DATA ##
#             ######################


#             input_beta_val = np.array([c_normalized[:, :, t-1] for t in range(train_size + 1, train_size + val_size + 1)])
#             r_t_val = np.array([val_data[:, t] for t in range(val_size)])


#             # input_beta_val = np.array([c_normalized[:, :, t] for t in range(train_size, train_size + val_size)])[:-1]
#             # r_t_val = np.array([val_data[:, t - train_size] for t in range(train_size, train_size + val_size)])

#             print(f'beta input VAL {input_beta_val.shape}')
#             print(f'r_t VAL {r_t_val.shape}')

#             # Ensure the shape is consistent
#             assert input_beta_val.shape[0] == r_t_val.shape[0], "Shape mismatch in validation data"

#             # Calculate the input for the factor network
#             input_factor_val = calculate_factor_network_input(input_beta_val, r_t_val)

#             #################
#             ### TEST DATA ##
#             #################

#             # Ensure that the input_beta_test and r_t_test ranges are correctly aligned
#             input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
#             r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])

#             print(f'beta input TEST {input_beta_test.shape}')

#             # Print shapes for debugging
#             print("Test shapes:", input_beta_test.shape, r_t_test.shape)

#             # Calculate the input for the factor network
#             input_factor_test = calculate_factor_network_input(input_beta_test, r_t_test)

#             # Ensure the shape is consistent
#             assert input_beta_test.shape[0] == r_t_test.shape[0], "Shape mismatch in test data"

#             best_val_R2_nn = -np.inf
#             best_l1_value = None

#             for l1_value in l1_values:
#                 # Build and train the neural network model with current l1_value
#                 main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)

#                 # Train the model with early stopping
#                 main_model.fit([input_beta_train,
#                     input_factor_train],  # Use train data directly
#                     r_t_train,  # Use train returns directly
#                     epochs=100,  # Set a high number of epochs
#                     batch_size=45,
#                     verbose=0,
#                     validation_data=([input_beta_val, input_factor_val], r_t_val),  # Use validation data directly
#                     callbacks=[early_stopping]
#                 )

#                 # Evaluate on validation data
#                 val_pred = main_model.predict([input_beta_val ,input_factor_val])
#                 val_R2_nn = np.mean(1 - np.var(r_t_val - val_pred, axis=1) / np.var(r_t_val, axis=1))

#                 # Select the best l1_value based on validation performance
#                 if val_R2_nn > best_val_R2_nn:
#                     best_val_R2_nn = val_R2_nn
#                     best_l1_value = l1_value

#             print('------------------------------------------------')
#             print(f'Best val R^2 {best_val_R2_nn}')

#             # Train final model with the best l1_value on the entire training data
#             main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)
#             main_model.fit([input_beta_train,
#                 input_factor_train],  # Use train data directly
#                 r_t_train,  # Use train returns directly
#                 epochs=100,  # Set a high number of epochs
#                 batch_size=45,
#                 verbose=0,
#                 validation_data=([input_beta_val, input_factor_val], r_t_val),  # Use validation data directly
#                 callbacks=[early_stopping]
#             )

#             # Evaluate on test data

#             # Evaluate on test data
#             test_pred = main_model.predict([input_beta_test, input_factor_test])

#             test_R2_nn = np.mean(1 - np.var(r_t_test - test_pred, axis=1) / np.var(r_t_test, axis=1))

#             print(f'TEST R^2 : {test_R2_nn}')

#             # Calculate predicted x_t
#             predicted_x_t = calculate_factor_network_input(input_beta_test, test_pred)

#             # Evaluate predicted x_t
#             true_x_t = input_factor_test
#             RSS = np.sum((true_x_t - predicted_x_t) ** 2)
#             TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
#             R2_x_t = 1 - (RSS / TSS)
#             print(f'R^2 for TEST x_t: {R2_x_t}')

#             # Calculate the Sharpe ratio for predicted x_t OOS
#             sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)
#             print('------------------------------------------------')
#             print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
#             print('------------------------------------------------')

#             # Calculate the portfolio return for each time period
#             portfolio_returns = np.zeros(r_t_test.shape[1])
#             for t in range(r_t_test.shape[1]):
#                 portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], r_t_test[:, t])

#             # Calculate the cumulative portfolio return
#             cumulative_portfolio_return = np.cumsum(portfolio_returns)

#             # Create a DataFrame for the portfolio returns
#             start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
#             date_range = pd.date_range(start_date, periods=r_t_test.shape[1], freq='M')
#             df_portfolio_returns = pd.DataFrame({
#                 'Date': date_range,
#                 'Portfolio_Return': portfolio_returns,
#                 'Cumulative_Portfolio_Return': cumulative_portfolio_return
#             })

#             long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

#             print('------------------------------------------------')
#             print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

#             print('------------------------------------------------')
#             print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
#             results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
#             print('------------------------------------------------')
#             if (i + 1) % 10 == 0:
#                 print(f'Simulation {i + 1}/{num_simulations} completed')

#             scores.append([str(num_hidden_layers), factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value, adjusted_R2])

#         # Convert scores list to DataFrame
#         scores_df = pd.DataFrame(scores, columns=cols)

#         # Save scores_df to HDF5 file for the current factor
#         scores_df.to_hdf(str(results_path / 'scores.h5'), key=f'{num_hidden_layers}_hidden_layers/{factor}', append=True, format='table')

# # Aggregate Results
# adjusted_R2_values = [res[0] for res in results]
# val_R2_nn_values = [res[1] for res in results]
# test_R2_nn_values = [res[2] for res in results]

# mean_adjusted_R2 = np.mean(adjusted_R2_values)
# std_adjusted_R2 = np.std(adjusted_R2_values)

# mean_val_R2_nn = np.mean(val_R2_nn_values)
# std_val_R2_nn = np.std(val_R2_nn_values)

# mean_test_R2_nn = np.mean(test_R2_nn_values)
# std_test_R2_nn = np.std(test_R2_nn_values)

# print(f'Mean Adjusted R2 over {num_simulations} simulations: {mean_adjusted_R2:.4f}')
# print(f'Standard Deviation of Adjusted R2: {std_adjusted_R2:.4f}')

# print(f'Mean Validation R2 (NN) over {num_simulations} simulations: {mean_val_R2_nn:.4f}')
# print(f'Standard Deviation of Validation R2 (NN): {std_val_R2_nn:.4f}')

# print(f'Mean Test R2 (NN) over {num_simulations} simulations: {mean_test_R2_nn:.4f}')
# print(f'Standard Deviation of Test R2 (NN): {std_test_R2_nn:.4f}')

# # Plotting the distribution of Adjusted R2
# plt.hist(adjusted_R2_values, bins=20, edgecolor='black')
# plt.title('Distribution of Adjusted R2')
# plt.xlabel('Adjusted R2')
# plt.ylabel('Frequency')
# plt.show()

# # Plotting the distribution of Validation R2 (NN)
# plt.hist(val_R2_nn_values, bins=20, edgecolor='black')
# plt.title('Distribution of Validation R2 (NN)')
# plt.xlabel('Validation R2 (NN)')
# plt.ylabel('Frequency')
# plt.show()

# # Plotting the distribution of Test R2 (NN)
# plt.hist(test_R2_nn_values, bins=20, edgecolor='black')
# plt.title('Distribution of Test R2 (NN)')
# plt.xlabel('Test R2 (NN)')
# plt.ylabel('Frequency')
# plt.show()


# **Non-linear model $\rightarrow$ TOTAL $R^2$**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from tqdm import tqdm
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dot
from tensorflow.keras.models import Model
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from pathlib import Path

!pip install tables


##########################################
## DEFINING THE PATH TO SAVE RESULTS IN ##
##########################################
from google.colab import drive
drive.mount('/content/drive')

# Define the base path for Google Drive
base_path = Path('/content/drive/My Drive/Tidy Finance ML dataset/simulations_CA_TOTAL_NON_LINEAR_x_t')

# Define the results path relative to the base path
results_path = base_path / 'results' / 'asset_pricing'

# Check if the path exists, if not, create it
if not results_path.exists():
    results_path.mkdir(parents=True)

#####################################################################################################
## IF RUNTIME DISCONNECTS, THEN LOAD SAVED SCORES AND KEEP ON TRAININ FROM WHERE THE CODE LEFT OFF ##
#####################################################################################################

# # Load the saved scores and determine completed configurations
# def load_scores(results_path, models, factors):
#     completed_configs = set()
#     for num_hidden_layers in models:
#         for factor in factors:
#             file_path = results_path / f'scores_{num_hidden_layers}_{factor}.h5'
#             if file_path.exists():
#                 completed_configs.add((num_hidden_layers, factor))
#     return completed_configs





# Define parameters
N = 200  # Number of assets
T = 180  # Time periods
Pc = 50  # Number of characteristics
Px = 50  # Number of factor components
num_factors = 5  # Number of factors
R2_target = 0.45  # Target R2 for calibration


####################################
## DEFINE DATA GENERATING PROCESS ##
####################################

def generate_data(N, T, Pc, Px, num_factors, R2_target):
    # Simulate characteristics
    rho = np.random.uniform(0.9, 1, Pc)
    c = np.zeros((N, Pc, T))
    for t in range(1, T):
        c[:, :, t] = rho * c[:, :, t-1] + np.random.normal(0, 1, (N, Pc))

    # Normalize characteristics
    c_normalized = 2 / (N + 1) * (c.argsort(axis=0).argsort(axis=0) - 1) - 1

    # Generate factors and errors
    x = np.random.normal(0.03, 0.12, (Px, T))
    eta = np.random.normal(0, 0.01, (num_factors, T))
    epsilon = np.random.standard_t(5, (N, T))

    # Define linear factor loadings
    beta_linear = np.zeros((N, num_factors, T))
    for t in range(1, T):
        beta_linear[:, :, t] = np.array([
                  np.power(1.2 * c_normalized[:, 0, t], 3),
                  c_normalized[:, 1, t] * c_normalized[:, 0, t],
                  np.power(0.4 * c_normalized[:, 2, t], 2),
                  2 * c_normalized[:, 3, t] * c_normalized[:, 2, t],
                  np.power(0.6 * c_normalized[:, 4, t], 4)
              ]).T

    # Initial Returns Calculation
    f = np.dot(np.random.randn(num_factors, Px), x) + eta
    r = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]) + epsilon[:, 1:]  # Slice epsilon to match dimensions

    # Calculate initial R2
    explained_variance = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f[:, 1:]), axis=1)
    total_variance = np.var(r, axis=1)
    initial_R2 = np.mean(explained_variance / total_variance)

    # Adjust the variances to achieve the target R2
    adjustment_factor = np.sqrt(R2_target / initial_R2)
    f_adjusted = adjustment_factor * f
    epsilon_adjusted = np.sqrt((1 - R2_target) / (1 - initial_R2)) * epsilon

    # Generate adjusted returns
    r_adjusted = np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]) + epsilon_adjusted[:, 1:]

    # Calculate adjusted R2
    explained_variance_adjusted = np.var(np.einsum('ijk,jk->ik', beta_linear[:, :, :-1], f_adjusted[:, 1:]), axis=1)
    total_variance_adjusted = np.var(r_adjusted, axis=1)
    adjusted_R2 = np.mean(explained_variance_adjusted / total_variance_adjusted)

    # Split data into training, validation, and testing sets
    # train_ratio = 0.6
    # val_ratio = 0.2
    # train_size = int(T * train_ratio)
    # val_size = int(T * val_ratio)
    # test_size = int(T*(1 - train_ratio - val_ratio)-1) # subtract the last time period because the characteristics are lagged

    # train_data = r_adjusted[:, :train_size]  # Training data up to train_size
    # val_data = r_adjusted[:, train_size:train_size+val_size]  # Validation data from train_size to train_size + val_size
    # test_data = r_adjusted[:, train_size+val_size:]  # Testing data from train_size + val_size to the end

    # Split data into training, validation, and testing sets
    train_ratio = 0.6
    val_ratio = 0.2
    train_size = int(T * train_ratio)
    val_size = int(T * val_ratio)
    test_size = T - train_size - val_size - 1  # subtract the last time period because the characteristics are lagged test size defines the size of the index of the returns.

    train_data = r_adjusted[:, :train_size]  # Training data up to train_size
    val_data = r_adjusted[:, train_size:train_size + val_size]  # Validation data from train_size to train_size + val_size
    test_data = r_adjusted[:, train_size + val_size :]  # Testing data from train_size + val_size to the end


    return train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size



###############################################################################
## Define CA model with a dynamic number of layers that can be iterated over ##
###############################################################################


def make_model(N, Pc, n_factors=6, l1_reg=0.01, num_hidden_layers=0, return_components=False):
    input_beta = Input((N, Pc), name='input_beta')
    input_factor = Input((Pc,), name='input_factor')

    x = input_beta

    if num_hidden_layers >= 1:
        # First hidden layer with 32 neurons and batch normalization
        hidden_layer_1 = Dense(units=32, activation='relu', name='hidden_layer_1', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_1')(hidden_layer_1)

    if num_hidden_layers >= 2:
        # Second hidden layer with 16 neurons and batch normalization
        hidden_layer_2 = Dense(units=16, activation='relu', name='hidden_layer_2', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_2')(hidden_layer_2)

    if num_hidden_layers >= 3:
        # Third hidden layer with 8 neurons and batch normalization
        hidden_layer_3 = Dense(units=8, activation='relu', name='hidden_layer_3', kernel_regularizer=l1(l1_reg))(x)
        x = BatchNormalization(name='batch_norm_3')(hidden_layer_3)

    # Output beta layer with L1 regularization
    output_beta = Dense(units=n_factors, name='output_beta', kernel_regularizer=l1(l1_reg))(x)

    # Output factor layer with L1 regularization
    output_factor = Dense(units=n_factors, name='output_factor', kernel_regularizer=l1(l1_reg))(input_factor)

    # Dot product layer combining output_beta and output_factor
    output = Dot(axes=(2, 1), name='output_layer')([output_beta, output_factor])

    # Compile the main model
    model = Model(inputs=[input_beta, input_factor], outputs=output)
    model.compile(loss='mse', optimizer='adam')

    # Create a standalone factor network model
    factor_model = Model(inputs=input_factor, outputs=output_factor)

    if return_components:
        return model, factor_model  # Return both the combined model and the factor network
    else:
        return model  # Return only the combined model by default

###########################################################################
## DEFINE HELPER FUNCTIONS FOR THE TRAINING LOOP TO MAKE IT MORE COMPACT ##
###########################################################################

# Function to calculate the input for the factor network
def calculate_factor_network_input(Z_t_minus_1, r_t):
    factor_input = []
    for t in range(Z_t_minus_1.shape[0]):
        Z_t = Z_t_minus_1[t]
        r_t_i = r_t[t]
        Z_t_inv = np.linalg.pinv(np.dot(Z_t.T, Z_t))
        factor_input_t = np.dot(Z_t_inv, np.dot(Z_t.T, r_t_i))
        factor_input.append(factor_input_t)
    return np.array(factor_input)

# Function for calculation of the sharpe ratio
def calculate_sharpe_ratio(returns, risk_free_rate=0):
    mean_return = np.mean(returns)
    std_dev_return = np.std(returns)
    sharpe_ratio = (mean_return - risk_free_rate) / std_dev_return
    return sharpe_ratio

# Function to calculate the long-short portfolio
def calculate_long_short_portfolio(predicted_returns, actual_returns, num_deciles=10):
    # Number of assets
    N = predicted_returns.shape[0]

    # Determine the threshold for the top and bottom deciles
    top_decile_threshold = np.percentile(predicted_returns, 100 * (1 - 1/num_deciles))
    bottom_decile_threshold = np.percentile(predicted_returns, 100 * 1/num_deciles)

    # Identify the long and short positions
    long_positions = predicted_returns >= top_decile_threshold
    short_positions = predicted_returns <= bottom_decile_threshold

    # Calculate the returns for the long and short positions
    long_returns = actual_returns[long_positions].mean(axis=0)
    short_returns = actual_returns[short_positions].mean(axis=0)

    # Calculate the portfolio return (long returns - short returns)
    portfolio_return = long_returns - short_returns

    return portfolio_return


#######################################
## DEFINE PARAMETERS TO ITERATE OVER ##
#######################################

# Define the different model configurations
models = [0, 1, 2, 3]  # Number of hidden layers
#
# Monte Carlo Simulation with L1 tuning and Early Stopping
num_simulations = 100
factors = [1, 2, 3, 4, 5, 6]  # Example factor list
#, 1,
# L1 regularization values to search over
l1_values = [0.0001, 0.001, 0.01, 0.1]
#, 0.001

# Early stopping callback
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


# Define columns
cols = ['Model', 'Factors', '$x_t$ Sharpe Ratio', 'Long-Short Portfolio Sharpe Ratio', '$R^2$ for $r_t$', 'Best L_1 Value', 'Adjusted $R^2$', '$R^2$ for $x_t$']

results = []  # Define results here
scores_dict = {} # Define scores_dict here
#######################################
## training loop with simulated data ##
#######################################

results = []  # Define results here

for num_hidden_layers in tqdm(models):
    for factor in tqdm(factors):
        scores = []
        for i in tqdm(range(num_simulations)):
            train_data, val_data, test_data, adjusted_R2, c_normalized, train_size, val_size, test_size = generate_data(N, T, Pc, Px, num_factors, R2_target)


            print(f'Train data{train_data.shape}')
            print(f'Val data shape :{val_data.shape}')
            print(f'Test data shape :{test_data.shape}')

            print(f'Train size:{train_size}')
            print(f'Val size :{val_size}')
            print(f'Test size :{test_size}')
            ################
            ## TRAIN DATA ##
            ################

            # Prepare inputs for the factor network input calculation
            input_beta_train = np.array([c_normalized[:, :, t-1] for t in range(1, train_size + 1)])

            r_t_train = np.array([train_data[:, t] for t in range(train_size)])


            print(input_beta_train.shape, r_t_train.shape)
            # Calculate the input for the factor network
            input_factor_train = calculate_factor_network_input(input_beta_train, r_t_train)

            # Ensure the shape is consistent
            assert input_beta_train.shape[0] == r_t_train.shape[0], "Shape mismatch in validation data"

            ######################
            ### VALIDATION DATA ##
            ######################

            input_beta_val = np.array([c_normalized[:, :, t-1] for t in range(train_size + 1, train_size + val_size + 1)])
            r_t_val = np.array([val_data[:, t] for t in range(val_size)])

            print(f'beta input VAL{input_beta_val.shape}')
            print(f'r_t VAL{r_t_val.shape}')

            # Calculate the input for the factor network
            input_factor_val = calculate_factor_network_input(input_beta_val, r_t_val)

            print(f'beta input VAL {input_beta_val.shape}')
            print(f'r_t VAL {r_t_val.shape}')

            # Ensure the shape is consistent
            assert input_beta_val.shape[0] == r_t_val.shape[0], "Shape mismatch in validation data"


            #################
            ### TEAST DATA ##
            #################


            # Ensure that the input_beta_test and r_t_test ranges are correctly aligned
            input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            r_t_test = np.array([test_data[:, t - (train_size + val_size + 1)] for t in range(train_size + val_size + 1, T)])

            # # Ensure that the input_beta_test and r_t_test ranges are correctly aligned
            # input_beta_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_size + 1, T)])
            # r_t_test = np.array([test_data[:, t ] for t in range(test_size)])

            print(f'beta input TEST {input_beta_test.shape}')

            # Print shapes for debugging
            print("Test shapes:", input_beta_test.shape, r_t_test.shape)

            # Calculate the input for the factor network
            input_factor_test = calculate_factor_network_input(input_beta_test, r_t_test)


            # Ensure the shape is consistent
            assert input_beta_test.shape[0] == r_t_test.shape[0], "Shape mismatch in test data"
            # Variables to track the best model and its performance
            best_val_R2_nn = -np.inf
            best_l1_value = None
            best_model_weights = None

            # Iterate over each L1 regularization value
            for l1_value in l1_values:
                # Build and train the neural network model with current l1_value
                main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=l1_value, num_hidden_layers=num_hidden_layers, return_components=True)

                # Train the model with early stopping
                main_model.fit(
                    [input_beta_train, input_factor_train],
                    r_t_train,  # Use appropriate train data slice
                    epochs=100,  # Set a high number of epochs
                    batch_size=45,
                    verbose=0,
                    validation_data=([input_beta_val, input_factor_val], r_t_val),  # Validation data is needed for early stopping
                    callbacks=[early_stopping]
                )

                # Evaluate on validation data
                val_pred = main_model.predict([input_beta_val, input_factor_val])
                val_R2_nn = np.mean(1 - np.var(r_t_val - val_pred, axis=1) / np.var(r_t_val, axis=1))

                # Select the best l1_value based on validation performance
                if val_R2_nn > best_val_R2_nn:
                    best_val_R2_nn = val_R2_nn
                    best_l1_value = l1_value
                    best_model_weights = main_model.get_weights()

            print('------------------------------------------------')
            print(f'Best val R^2: {best_val_R2_nn} with L1 regularization: {best_l1_value}')

            # Use the best L1 regularization value to initialize the final model
            main_model, factor_network_model = make_model(N, Pc, n_factors=num_factors, l1_reg=best_l1_value, num_hidden_layers=num_hidden_layers, return_components=True)

            # Load the best model weights
            main_model.set_weights(best_model_weights)

            # Evaluate on test data without retraining
            test_pred = main_model.predict([input_beta_test, input_factor_test])

            test_R2_nn = np.mean(1 - np.var(r_t_test - test_pred, axis=1) / np.var(r_t_test, axis=1))

            print(f'TEST R^2 : {test_R2_nn}')
            # Calculate predicted x_t
            predicted_x_t = calculate_factor_network_input(input_beta_test, test_pred)
            # print(f'PREDICTED XT {predicted_x_t.shape}')
            # print(f'TRUE XT {test_factor_network_input.shape}')

            # Evaluate predicted x_t
            true_x_t = input_factor_test
            RSS = np.sum((true_x_t - predicted_x_t) ** 2)
            TSS = np.sum((true_x_t - np.mean(true_x_t)) ** 2)
            R2_x_t = 1 - (RSS / TSS)
            print(f'R^2 for TEST x_t: {R2_x_t}')

            # Calculate the Sharpe ratio for predicted x_t OOS
            sharpe_ratio_x_t = calculate_sharpe_ratio(predicted_x_t)
            print('------------------------------------------------')
            print(f'Sharpe Ratio for predicted x_t: {sharpe_ratio_x_t}')
            print('------------------------------------------------')

            # Calculate the portfolio return for each time period
            portfolio_returns = np.zeros(r_t_test.shape[1])
            for t in range(r_t_test.shape[1]):
                portfolio_returns[t] = calculate_long_short_portfolio(test_pred[:, t], r_t_test[:, t])

            # Calculate the cumulative portfolio return
            cumulative_portfolio_return = np.cumsum(portfolio_returns)

            # Create a DataFrame for the portfolio returns
            start_date = pd.Timestamp('2000-01-01')  # Change to your desired start date
            date_range = pd.date_range(start_date, periods=r_t_test.shape[1], freq='M')
            df_portfolio_returns = pd.DataFrame({
                'Date': date_range,
                'Portfolio_Return': portfolio_returns,
                'Cumulative_Portfolio_Return': cumulative_portfolio_return
            })

            long_short_portfolio_sharpe_ratio = calculate_sharpe_ratio(portfolio_returns)

            print('------------------------------------------------')
            print(f'Sharpe ratio for long-short portfolio returns : {long_short_portfolio_sharpe_ratio}')

            print('------------------------------------------------')
            print(f'Iteration : {i}|TEST R^2 : {test_R2_nn}')
            results.append((adjusted_R2, best_val_R2_nn, test_R2_nn))
            print('------------------------------------------------')
            if (i + 1) % 10 == 0:
                print(f'Simulation {i + 1}/{num_simulations} completed')
            scores.append([str(num_hidden_layers) ,factor, sharpe_ratio_x_t, long_short_portfolio_sharpe_ratio, test_R2_nn, best_l1_value, adjusted_R2, R2_x_t])


        # Convert scores list to DataFrame
        scores_df = pd.DataFrame(scores, columns=cols)
        scores_dict[(num_hidden_layers, factor)] = scores_df

        scores_df.to_hdf(str(results_path / f'scores_{num_hidden_layers}_{factor}.h5'), key='scores', mode='w')



# Plotting the results for each model and factor
for (num_hidden_layers, factor), df in scores_dict.items():
    plt.figure(figsize=(20, 5))  # Adjusted figure size to accommodate the extra subplot

    plt.subplot(1, 5, 1)
    plt.hist(df['Adjusted $R^2$'], bins=20, edgecolor='black')
    plt.title(f'Adjusted $R^2$ from DGP, CA Factor {factor}')
    plt.xlabel('Adjusted $R^2$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 2)
    plt.hist(df['$R^2$ for $x_t$'], bins=20, edgecolor='black')
    plt.title(f'$R^2$ for $x_t$, $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Adjusted $R^2$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 3)
    plt.hist(df['$R^2$ for $r_t$'], bins=20, edgecolor='black')
    plt.title(f'$R^2$ for $r_t$, $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Test $R^2$')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 4)
    plt.hist(df['$x_t$ Sharpe Ratio'], bins=20, edgecolor='black')
    plt.title(f'Sharpe Ratio $x_t$ for Model $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('$x_t$ Sharpe Ratio')
    plt.ylabel('Frequency')

    plt.subplot(1, 5, 5)  # Added subplot for the long-short portfolio Sharpe ratio
    plt.hist(df['Long-Short Portfolio Sharpe Ratio'], bins=20, edgecolor='black')
    plt.title(f'Long-Short Portfolio SR for $CA_{num_hidden_layers}$, Factor {factor}')
    plt.xlabel('Long-Short Portfolio SR')
    plt.ylabel('Frequency')

    plt.tight_layout()

    # Save the plot
    plot_filename = results_path / f'model_{num_hidden_layers}_factor_{factor}.png'
    plt.savefig(plot_filename)
    plt.close()



In [ ]:
print(test_size)

In [ ]:
print(test_r_t)

In [ ]:
print(r_t_train.shape, input_beta_train.shape, input_factor_train.shape)


In [ ]:
print(test_r_t.shape)

In [ ]:
print(test_input_beta.shape[0] ,test_r_t.shape[0])

In [ ]:
print(train_data[:, 1:train_size].T.shape)

In [ ]:
print(input_beta_train.shape, input_factor_train.shape)

In [ ]:

print(val_input_beta.shape, val_factor_network_input.shape, val_data[:, 1:train_size].T.shape)

print(input_beta_train.shape,input_factor_train.shape, train_data[:, 1:train_size].T.shape )


In [ ]:

print(Z_t_minus_1_test.shape)
print(test_data.shape[1])

In [ ]:
print(val_r_t.shape[0])
print(val_input_beta.shape[0])

In [ ]:
# Prepare test inputs
Z_t_minus_1_test = np.array([c_normalized[:, :, t-1] for t in range(train_size + val_data.shape[1], T)])
test_input_beta = Z_t_minus_1_test.reshape(test_data.shape[1]-1, N, Pc)

test_r_t = test_data[:, :-1].T
